<a href="https://colab.research.google.com/github/joaoslino50/kl-hybrid-gcr/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph-Constrained Reasoning for Reliable LLM Question Answering (`kl-hybrid-gcr`)

**Google Colab Runner** — Open directly from GitHub (`File -> Open notebook -> GitHub -> joaoslino50/kl-hybrid-gcr`).

In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive so outputs survive Colab disconnects
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/kl-hybrid-gcr-outputs"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# 2. Clone or update the repository
if not os.path.exists("/content/kl-hybrid-gcr"):
    !git clone https://github.com/joaoslino50/kl-hybrid-gcr.git
%cd /content/kl-hybrid-gcr
!git pull origin main
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into 'kl-hybrid-gcr'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 164 (delta 73), reused 138 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.14 MiB | 27.75 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/kl-hybrid-gcr
From https://github.com/joaoslino50/kl-hybrid-gcr
 * branch            main       -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
from getpass import getpass
from huggingface_hub import login

# Prompts you with a hidden password box so your token is never saved in plain text in the .ipynb file
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Paste your HF_TOKEN: ").strip()

login(token=os.environ["HF_TOKEN"])
!python scripts/verify_setup.py

Paste your HF_TOKEN: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GCR-PCST-KGQA Backbone Setup Verification

[1/3] Checking core package versions...
  - python                : 3.13.16
  - torch                 : 2.14.1+cu130
  - transformers          : 5.22.1
  - sentence_transformers : 5.7.0
  - networkx              : 3.7
  - pcst_fast             : pcst_fast (loaded)

[2/3] Detecting compute device and running matrix multiplication...
2026-11-17 19:28:32,290 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA A100-SXM4-80GB)
  - Active device         : cuda
  - MatMul output shape   : (128, 128) (norm=1448.4880)

[3/3] Testing SubgraphData, NetworkX conversion, KGTrie & KGTrieLogitsProcessor...
  - SubgraphData -> DiGraph: 4 nodes, 3 edges
  - Allowed next tokens   : [20, 25]
  - Unmasked logit tokens : [20, 25] (all invalid tokens masked to -inf)

SUCCESS: All environment, device, and KG-Trie interface checks passed!


## Checkpoint 1 Evaluations (PCST Retrieval, Baselines, and Live Wikidata Demo)

In [3]:
# 1. PCST Retrieval evaluation on 200 WebQSP samples (SentenceBERT + pcst_fast)
!python scripts/eval_retrieval.py \
    --source webqsp \
    --max-samples 200 \
    --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/retrieval_webqsp_200.jsonl

PCST params: topk_nodes=50 topk_edges=50 edge_cost=0.2 anchor_prize=4.0
README.md: 100% 900/900 [00:00<00:00, 3.63MB/s]
modules.json: 100% 349/349 [00:00<00:00, 1.38MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 542kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 1.24MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 256kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.79MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:00<00:00, 130MB/s, 6.94MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:00<00:00, 86.4MB/s, 8.24MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:00<00:00, 92.3MB/s, 8.85MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 7114.96it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.36MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 38.6MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 35.2MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 608kB/s]
config.json: 100% 190/190 [00:00

In [4]:
# 2. Crash-safe Checkpoint 1 baselines on 100 WebQSP samples (Llama-3-8B)
!python scripts/eval_baselines.py --mode llm_only_zero_shot --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/llm_zs_100.jsonl
!python scripts/eval_baselines.py --mode llm_only_cot --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/llm_cot_100.jsonl
!python scripts/eval_baselines.py --mode standard_rag --max-samples 100 --resume --out /content/drive/MyDrive/kl-hybrid-gcr-outputs/rag_100.jsonl

Resuming: 100 questions already in /content/drive/MyDrive/kl-hybrid-gcr-outputs/llm_zs_100.jsonl
Device: cuda | mode: llm_only_zero_shot | source: webqsp/test
config.json: 100% 654/654 [00:00<00:00, 2.89MB/s]
tokenizer_config.json: 100% 51.0k/51.0k [00:00<00:00, 103MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 20.1MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 292kB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 50.3MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   4% 604M/16.1G [00:02<00:45, 341MB/s, 99.2MB/s  ]
Reconstructing (incomplete total...):  11% 1.79G/16.1G [00:03<00:17, 793MB/s,  707MB/s  ]
Reconstructing 

In [5]:
# 3. Live Wikidata SPARQL acquisition + Hybrid pipeline + Interactive HTML graph export
# Cell 5 in Colab:
!git pull origin main
!python scripts/run_pipeline.py \
    --source wikidata \
    --mode hybrid \
    --export-html /content/drive/MyDrive/kl-hybrid-gcr-outputs/wikidata_demo.html

From https://github.com/joaoslino50/kl-hybrid-gcr
 * branch            main       -> FETCH_HEAD
Already up to date.
2026-11-17 19:34:08,954 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA A100-SXM4-80GB)
2026-11-17 19:34:08,955 | INFO | src.acquisition.sparql_client | Fetching Wikidata hop 1/2 for 1 frontier entities
2026-11-17 19:34:09,099 | WARNING | src.acquisition.sparql_client | Wikidata SPARQL hop 1 rate-limited or unreachable (HTTP Error 429: Aggressively rate-limiting to 1 req / min - this rule was created during active wdqs outage (797a132)); keeping 0 triples.
2026-11-17 19:34:09,099 | WARNING | src.acquisition.sparql_client | Wikidata endpoint rate-limited Colab IP (HTTP 429); using cached Wikidata Q45 neighborhood.
2026-11-17 19:34:13,124 | INFO | src.retrieval.embedder | Loading SentenceBERT model 'sentence-transformers/all-MiniLM-L6-v2' on cuda
2026-11-17 19:34:13,278 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-Mi

## Checkpoint 2 / Full Hybrid vs. All-Modes Pipeline

In [6]:
!python scripts/run_pipeline.py \
    --source webqsp \
    --mode hybrid \
    --max-samples 5 \
    --output-json /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_hybrid_5.json \
    --export-html /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_sample0.html

2026-11-17 19:34:31,464 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA A100-SXM4-80GB)
2026-11-17 19:34:31,464 | INFO | src.acquisition.dataset_loader | Loading dataset 'rmanluo/RoG-webqsp' (split='test', streaming=True)
2026-11-17 19:34:31,632 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-17 19:34:31,642 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rmanluo/RoG-webqsp/c0632533135a06f8c5d536b420deec5fcb5c58f3/README.md?%2Fdatasets%2Frmanluo%2FRoG-webqsp%2Fresolve%2Fmain%2FREADME.md=&etag=%22ffc0eb5b1125d866311eddac4f8d339a37c87cd0%22 "HTTP/1.1 200 OK"
2026-11-17 19:34:31,752 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py "HTTP/1.1 404 Not Found"
2026-11-17 19:34:32,050 | INFO | httpx | HTTP Request: HEAD https://s3.amazona

In [7]:
!python scripts/run_pipeline.py \
    --source webqsp \
    --mode all \
    --max-samples 5 \
    --output-json /content/drive/MyDrive/kl-hybrid-gcr-outputs/webqsp_all_modes_5.json

2026-11-17 19:35:09,402 | INFO | src.utils.device | Selected compute device: cuda (NVIDIA A100-SXM4-80GB)
2026-11-17 19:35:09,402 | INFO | src.acquisition.dataset_loader | Loading dataset 'rmanluo/RoG-webqsp' (split='test', streaming=True)
2026-11-17 19:35:09,555 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-17 19:35:09,566 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rmanluo/RoG-webqsp/c0632533135a06f8c5d536b420deec5fcb5c58f3/README.md?%2Fdatasets%2Frmanluo%2FRoG-webqsp%2Fresolve%2Fmain%2FREADME.md=&etag=%22ffc0eb5b1125d866311eddac4f8d339a37c87cd0%22 "HTTP/1.1 200 OK"
2026-11-17 19:35:09,669 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py "HTTP/1.1 404 Not Found"
2026-11-17 19:35:09,962 | INFO | httpx | HTTP Request: HEAD https://s3.amazona

In [8]:
from google.colab import runtime
print("Experiment complete! Disconnecting A100 runtime to save Compute Units...")
runtime.unassign()

Experiment complete! Disconnecting A100 runtime to save Compute Units...
